# Notebook for Downloading Events of a Season

### Imports

In [1]:
import logging
import csv
from typing import Any, Dict, List
import json
import json
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

from sportradar_datacore_api.handball import HandballAPI

In [2]:
load_dotenv()  # Load environment variables from .env file if present


True

### Configuration

In [3]:
NAME_COMPETITION = "1. Handball-Bundesliga"

NAME_SEASON = "DAIKIN HBL 2024/25"
YEAR_SEASON = int(NAME_SEASON.split()[-1].split("/")[0])
YEARS_SEASON = NAME_SEASON.split()[-1].replace("/", "-")

PATH_TO_OUTPUT = os.path.join(
    os.getcwd(), "..", "data", "season_24_25"
)

# create path if it does not exist
os.makedirs(PATH_TO_OUTPUT, exist_ok=True)

In [4]:
import duckdb
con = duckdb.connect(f'../data/mydb{YEARS_SEASON}.duckdb')

### Initialize API

In [5]:
api = HandballAPI(
    base_url=os.getenv("BASE_URL", ""),
    auth_url=os.getenv("AUTH_URL", ""),
    client_id=os.getenv("CLIENT_ID", ""),
    client_secret=os.getenv("CLIENT_SECRET", ""),
    org_id=os.getenv("CLIENT_ORGANIZATION_ID"),
    scopes=["read:organization"],
    sport="handball",
)

### Get wanted competition ID

In [6]:
id_competition = api.get_competition_id_by_name(NAME_COMPETITION)

# id_competition = int(id_competition)

# Check if the competition was found
if not id_competition:
    raise ValueError(f"Competition '{NAME_COMPETITION}' not found.")
else:
    print(f"→ Competition '{NAME_COMPETITION}' -> {id_competition}")

id_season = api.get_season_id_by_year(
    competition_id=id_competition, season_year=YEAR_SEASON
)
# Check if the season was found
if not id_season:
    raise ValueError(f"Season '{NAME_SEASON}' not found in competition '{NAME_COMPETITION}'.")
else:
    print(f"→ Season '{NAME_SEASON}' -> {id_season}")

→ Competition '1. Handball-Bundesliga' -> 4c445e5c-3956-11ef-9d0e-b74f5c057367
→ Season 'DAIKIN HBL 2024/25' -> cabcf509-4373-11ef-a370-9d3c1e90234a


# Get Teams in the Season

In [7]:
list_entities_season = api.get_teams_by_season_id(season_id=id_season)
# display( pd.json_normalize(list_entities_season[0].to_dict()) )
print(f"→ Number of teams in season '{NAME_SEASON}': {len(list_entities_season)}")

→ Number of teams in season 'DAIKIN HBL 2024/25': 18


### Cols to keep from get_team_by_id

In [8]:
columns_to_keep = [
    "entityId",
    # "organizationId",
    "organization",
    # "entityGroupId",
    # "entityGroup",
    # "internationalReference",
    # "status",
    "nameFullLocal",
    # "additionalNames",
    "nameFullLatin",
    "codeLocal",
    "codeLatin",
    # "address",
    # "social",
    # "contacts",
    # "colors",
    # "historicalNames",
    "externalId",
    # "ageGroup",
    # "gender",
    # "standard",
    # "grade",
    # "representing",
    # "discipline",
    # "updated",
    # "added",
    # "defaultVenueId",
    # "alternateVenueIds",
    # "images"
]

In [9]:
df_teams = pd.DataFrame()

# print len of list_entities_season
print(f"Number of teams in season: {len(list_entities_season)}")

for team in list_entities_season:
    id_team = team.entity_id
    team_details = api.get_team_by_id(entity_id=id_team)
    # print all keys of team_details[0]
    # print("Keys in team_details[0]:")
    # print(json.dumps(list(team_details[0].to_dict().keys()), indent=4, default=str))
    # convert to dataframe
    df_team_details = pd.json_normalize(team_details[0].to_dict())
    # display(entity_details)

    # drop columns that are not in columns list
    df_team_details = df_team_details[[col for col in columns_to_keep if col in df_team_details.columns]]
    # display(df_team_details)
    df_teams = pd.concat([df_teams, df_team_details], ignore_index=True)

    # as json dump
    # print(json.dumps(team_details[0].to_dict(), indent=4, default=str))

Number of teams in season: 18


In [10]:
# Drop tables if exist
con.execute("DROP TABLE IF EXISTS teams")
# Create duckdb table
con.execute("""
CREATE TABLE IF NOT EXISTS teams AS SELECT * FROM df_teams
""")

In [11]:
# plot table
con.execute("SELECT * FROM teams").df()

,entityId,nameFullLocal,nameFullLatin,codeLocal,codeLatin,externalId
0,fef51771-3952-11ef-97b4-af5c55c3771d,ThSV Eisenach,ThSV Eisenach,EIS,EIS,32
1,feace61d-3952-11ef-ae23-af5c55c3771d,SC DHfK Leipzig,SC DHfK Leipzig,LEI,LEI,13
2,febb3114-3952-11ef-b6f7-af5c55c3771d,TVB Stuttgart,TVB Stuttgart,TVB,TVB,17
3,feb41b46-3952-11ef-9df6-af5c55c3771d,MT Melsungen,MT Melsungen,MTM,MTM,15
4,fe911367-3952-11ef-9131-af5c55c3771d,VfL Gummersbach,VfL Gummersbach,GUM,GUM,7
5,fe8d1885-3952-11ef-9130-af5c55c3771d,HC Erlangen,HC Erlangen,HCE,HCE,6
6,fea93a14-3952-11ef-a7e0-af5c55c3771d,TSV Hannover-Burgdorf,,HAN,HAN,12
7,fe73c376-3952-11ef-8a18-af5c55c3771d,HSG Wetzlar,HSG Wetzlar,WET,WET,1
8,ffe84692-3952-11ef-954e-af5c55c3771d,1. VfL Potsdam,1. VfL Potsdam,POT,POT,93
9,febf038e-3952-11ef-b7c2-af5c55c3771d,THW Kiel,THW Kiel,THW,THW,18


In [12]:
columns_to_keep_fixtures = [
    "fixtureId",
    # "organizationId",
    # "organization",
    "seasonId",
    # "season",
    # "practiceDrillType",
    # "internationalReference",
    # "status",
    "fixtureNumber",
    "nameLocal",
    "nameLatin",
    "startTimeLocal",
    "startTimeUTC",
    # "startTimeActualUTC",
    # "endTimeActualUTC",
    # "timesUnconfirmed",
    # "locked",
    # "placingIfWon",
    # "placingIfLost",
    # "attendance",
    # "sellout",
    # "duration",
    # "durationFull",
    # "ticketURL",
    # "stageCode",
    # "stage",
    # "seriesCode",
    # "poolCode",
    # "roundCode",
    # "round",
    "roundNumber",
    # "liveDataAvailable",
    # "liveVideoAvailable",
    # "fixtureType",
    # "maximumPeriodTypeUsed",
    # "competitorType",
    "competitors",
    # "venueId",
    # "venue",
    "externalId",
    # "profileId",
    # "includeInStandings",
    # "updated",
    # "added",
    # "seriesFixtureNumber",
    # "discipline",
    # "broadcasts"
]

columns_to_keep_competitors = [
    "entityId",
    # "conferenceId",
    # "divisionId",
    # "includeInConferenceStatistics",
    "isHome",
    # "includeInRepresentation",
    "draw",
    # "resultStatus",
    "resultPlace",
    # "resultSecondaryScorePlace",
    # "startingNumber",
    "score",
    # "secondaryScore",
    # "shootOutAttempts",
    # "rosterStatus",
    # "isNeutralVenue",
    # "uniformId",
    # "externalId",
]

### Get the fixtures (matches) of a season and insert to duckdb

In [13]:
list_fixtures = api.get_list_matches_by_season_id(season_id=id_season)

print(f"Found {len(list_fixtures)} fixtures.")

# drop table fixtures if exists
con.execute("DROP TABLE IF EXISTS fixtures")

for match in list_fixtures:
    df_fixture = pd.json_normalize(match.to_dict())
    # drop columns that are not in columns list
    df_fixture = df_fixture[
        [col for col in columns_to_keep_fixtures if col in df_fixture.columns]
    ]

    competitors_expanded = pd.json_normalize(
        df_fixture["competitors"].explode().to_list()
    )
    # drop columns that are not in columns list
    competitors_expanded = competitors_expanded[
        [col for col in columns_to_keep_competitors if col in competitors_expanded.columns]
    ]    
    # insert names to competitors_expanded
    competitors_expanded = competitors_expanded.merge(
        df_teams[["entityId", "nameFullLocal"]],
        left_on="entityId",
        right_on="entityId",
        how="left",
    )
    # display(competitors_expanded)
    # convert competitors_expanded to json and add to df_fixture
    df_fixture = df_fixture.drop(columns=["competitors"])
    df_fixture = df_fixture.assign(competitors= [competitors_expanded.to_dict(orient="records")])
    # insert entityId_home and entityId_away to df_fixture
    df_fixture = df_fixture.assign(
        entityId_home=competitors_expanded[competitors_expanded["isHome"] == True]["entityId"].values[0],
        entityId_away=competitors_expanded[competitors_expanded["isHome"] == False]["entityId"].values[0],
    )
    # insert name_team_home and name_team_away to df_fixture
    df_fixture = df_fixture.assign(
        name_team_home=competitors_expanded[competitors_expanded["isHome"] == True]["nameFullLocal"].values[0],
        name_team_away=competitors_expanded[competitors_expanded["isHome"] == False]["nameFullLocal"].values[0],
    )
    # insert score_home and score_away to df_fixture
    df_fixture = df_fixture.assign(
        score_home=competitors_expanded[competitors_expanded["isHome"] == True]["score"].values[0],
        score_away=competitors_expanded[competitors_expanded["isHome"] == False]["score"].values[0],
    )
    # insert resultPlace_home and resultPlace_away to df_fixture
    df_fixture = df_fixture.assign(
        resultPlace_home=competitors_expanded[competitors_expanded["isHome"] == True]["resultPlace"].values[0],
        resultPlace_away=competitors_expanded[competitors_expanded["isHome"] == False]["resultPlace"].values[0],
    )


    # display(df_fixture)
    # break
    # append to duckdb table
    con.execute(
        """
    CREATE TABLE IF NOT EXISTS fixtures AS SELECT * FROM df_fixture
    """
    )
    con.execute(
        """
    INSERT INTO fixtures SELECT * FROM df_fixture
    """
    )



# plot duplicate fixtureid's
display(
    con.execute(
        """
SELECT fixtureId, COUNT(*) as count FROM fixtures
GROUP BY fixtureId
HAVING count > 1
"""
    ).df()
)

# drop duplicate fixtureid's keeping first
con.execute(
    """
DELETE FROM fixtures
WHERE rowid NOT IN (
    SELECT MIN(rowid)
    FROM fixtures
    GROUP BY fixtureId
)
"""
)
# plot table
display(con.execute("SELECT * FROM fixtures").df())


Found 306 fixtures.


,fixtureId,count
0,00c08679-4374-11ef-80bd-73cf0bc66b45,2


,fixtureId,seasonId,fixtureNumber,nameLocal,nameLatin,startTimeLocal,startTimeUTC,roundNumber,externalId,competitors,entityId_home,entityId_away,name_team_home,name_team_away,score_home,score_away,resultPlace_home,resultPlace_away
0,00c08679-4374-11ef-80bd-73cf0bc66b45,cabcf509-4373-11ef-a370-9d3c1e90234a,61,TSV Hannover-Burgdorf vs. SG Flensburg-Handewitt,<NA>,2024-10-20T15:00:00,2024-10-20T13:00:00,7,57979,[{'entityId': 'fe88f93b-3952-11ef-aa5d-af5c55c...,fea93a14-3952-11ef-a7e0-af5c55c3771d,fe88f93b-3952-11ef-aa5d-af5c55c3771d,TSV Hannover-Burgdorf,SG Flensburg-Handewitt,31,30,1,2
1,00febf5a-4374-11ef-9a3c-a3f6150225cd,cabcf509-4373-11ef-a370-9d3c1e90234a,62,SC Magdeburg vs. SC DHfK Leipzig,<NA>,2024-10-20T16:00:00,2024-10-20T14:00:00,7,57980,[{'entityId': 'fe848316-3952-11ef-8185-af5c55c...,fe848316-3952-11ef-8185-af5c55c3771d,feace61d-3952-11ef-ae23-af5c55c3771d,SC Magdeburg,SC DHfK Leipzig,35,29,1,2
2,0182fb4e-4374-11ef-a879-691708cc0833,cabcf509-4373-11ef-a370-9d3c1e90234a,63,TBV Lemgo Lippe vs. TVB Stuttgart,<NA>,2024-10-20T16:30:00,2024-10-20T14:30:00,7,57981,[{'entityId': 'fe99a935-3952-11ef-9dd4-af5c55c...,fe99a935-3952-11ef-9dd4-af5c55c3771d,febb3114-3952-11ef-b6f7-af5c55c3771d,TBV Lemgo Lippe,TVB Stuttgart,28,24,1,2
3,037982a4-4374-11ef-982a-1f74ee999933,cabcf509-4373-11ef-a370-9d3c1e90234a,64,SC DHfK Leipzig vs. MT Melsungen,<NA>,2024-10-24T19:00:00,2024-10-24T17:00:00,8,57982,[{'entityId': 'feace61d-3952-11ef-ae23-af5c55c...,feace61d-3952-11ef-ae23-af5c55c3771d,feb41b46-3952-11ef-9df6-af5c55c3771d,SC DHfK Leipzig,MT Melsungen,27,28,2,1
4,03a7d8f5-4374-11ef-abc8-75b52ec025f2,cabcf509-4373-11ef-a370-9d3c1e90234a,65,Handball Sport Verein Hamburg vs. TSV Hannover...,<NA>,2024-10-24T19:00:00,2024-10-24T17:00:00,8,57983,[{'entityId': '0045fbc4-3953-11ef-a217-af5c55c...,0045fbc4-3953-11ef-a217-af5c55c3771d,fea93a14-3952-11ef-a7e0-af5c55c3771d,Handball Sport Verein Hamburg,TSV Hannover-Burgdorf,32,32,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,fd35640d-4373-11ef-96cf-3d96bea3d744,cabcf509-4373-11ef-a370-9d3c1e90234a,56,Rhein-Neckar Löwen vs. HC Erlangen,<NA>,2024-10-17T19:00:00,2024-10-17T17:00:00,7,57974,[{'entityId': 'fe80598a-3952-11ef-914c-af5c55c...,fe80598a-3952-11ef-914c-af5c55c3771d,fe8d1885-3952-11ef-9130-af5c55c3771d,Rhein-Neckar Löwen,HC Erlangen,38,33,1,2
302,fd5a003d-4373-11ef-9920-89956fae12b0,cabcf509-4373-11ef-a370-9d3c1e90234a,57,VfL Gummersbach vs. ThSV Eisenach,<NA>,2024-10-18T19:00:00,2024-10-18T17:00:00,7,57975,[{'entityId': 'fe911367-3952-11ef-9131-af5c55c...,fe911367-3952-11ef-9131-af5c55c3771d,fef51771-3952-11ef-97b4-af5c55c3771d,VfL Gummersbach,ThSV Eisenach,34,32,1,2
303,fef1685c-4373-11ef-94c0-099f83d5fd51,cabcf509-4373-11ef-a370-9d3c1e90234a,59,MT Melsungen vs. Füchse Berlin,<NA>,2024-10-19T19:00:00,2024-10-19T17:00:00,7,57977,[{'entityId': 'fe7bdd16-3952-11ef-b585-af5c55c...,feb41b46-3952-11ef-9df6-af5c55c3771d,fe7bdd16-3952-11ef-b585-af5c55c3771d,MT Melsungen,Füchse Berlin,33,31,1,2
304,ff1398a1-4373-11ef-92f0-fdc5f2a62d4e,cabcf509-4373-11ef-a370-9d3c1e90234a,58,FRISCH AUF! Göppingen vs. SG BBM Bietigheim,<NA>,2024-10-18T20:00:00,2024-10-18T18:00:00,7,57976,[{'entityId': 'fea4237d-3952-11ef-9fcd-af5c55c...,fea4237d-3952-11ef-9fcd-af5c55c3771d,ff49c9b8-3952-11ef-b8c6-af5c55c3771d,FRISCH AUF! Göppingen,SG BBM Bietigheim,30,25,1,2


In [14]:
df_all_fixtures_in_season = con.execute("SELECT * FROM fixtures").df()

# Drop unused column
if "competitors" in df_all_fixtures_in_season.columns:
    df_all_fixtures_in_season = df_all_fixtures_in_season.drop(columns=["competitors"])

# --- prep & sorting ---
import numpy as np
import pandas as pd

for col in ["score_home", "score_away"]:
    if col in df_all_fixtures_in_season.columns:
        df_all_fixtures_in_season[col] = pd.to_numeric(df_all_fixtures_in_season[col], errors="coerce")

df_all_fixtures_in_season["startTimeUTC"] = pd.to_datetime(
    df_all_fixtures_in_season["startTimeUTC"], errors="coerce"
)

sort_cols = ["startTimeUTC"]
if "fixtureNumber" in df_all_fixtures_in_season.columns:
    sort_cols.append("fixtureNumber")
sort_cols.append("fixtureId")

df_all_fixtures_in_season = df_all_fixtures_in_season.sort_values(sort_cols).reset_index(drop=True)

# --- initialize standings state ---
team_id_cols = ["entityId_home", "entityId_away"]
teams = pd.unique(pd.concat([df_all_fixtures_in_season[c] for c in team_id_cols], ignore_index=True))

name_lookup = {}
if "name_team_home" in df_all_fixtures_in_season.columns and "name_team_away" in df_all_fixtures_in_season.columns:
    home_names = df_all_fixtures_in_season.set_index("entityId_home")["name_team_home"]
    away_names = df_all_fixtures_in_season.set_index("entityId_away")["name_team_away"]
    name_lookup = pd.concat([home_names, away_names]).dropna().groupby(level=0).first().to_dict()

# Add wins/draws/losses + pts_against to the per-team state
standings = {
    tid: {
        "pts": 0,           # points earned by the team (2/1/0)
        "pts_against": 0,   # points opponents earned vs this team
        "wins": 0,
        "draws": 0,
        "losses": 0,
        "gf": 0,
        "ga": 0,
        "gd": 0,
        "name": name_lookup.get(tid, "")
    }
    for tid in teams
}

def points_for_pair(h, a):
    if h > a:
        return 2, 0
    if h < a:
        return 0, 2
    return 1, 1  # draw

def standings_table_df():
    tbl = (
        pd.DataFrame.from_dict(standings, orient="index")
        .assign(gd=lambda x: x["gf"] - x["ga"])
    )
    tbl = tbl.sort_values(
        by=["pts", "gd", "gf", "name"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    tbl["rank"] = range(1, len(tbl) + 1)
    return tbl

# --- iterate fixtures and capture standings after each game ---
standing_after_home, standing_after_away = [], []

# New per-fixture cumulative outputs
wins_home_after, draws_home_after, losses_home_after, pts_against_home_after = [], [], [], []
wins_away_after, draws_away_after, losses_away_after, pts_against_away_after = [], [], [], []

for _, row in df_all_fixtures_in_season.iterrows():
    home_id = row["entityId_home"]
    away_id = row["entityId_away"]
    sh = row["score_home"]
    sa = row["score_away"]

    # if unplayed, just snapshot current ranks & cumulative stats
    if pd.isna(sh) or pd.isna(sa):
        tbl = standings_table_df()
        standing_after_home.append(tbl.loc[home_id, "rank"] if home_id in tbl.index else np.nan)
        standing_after_away.append(tbl.loc[away_id, "rank"] if away_id in tbl.index else np.nan)

        # copy current cumulative values
        wins_home_after.append(standings[home_id]["wins"])
        draws_home_after.append(standings[home_id]["draws"])
        losses_home_after.append(standings[home_id]["losses"])
        pts_against_home_after.append(standings[home_id]["pts_against"])

        wins_away_after.append(standings[away_id]["wins"])
        draws_away_after.append(standings[away_id]["draws"])
        losses_away_after.append(standings[away_id]["losses"])
        pts_against_away_after.append(standings[away_id]["pts_against"])
        continue

    sh, sa = int(sh), int(sa)

    # goals
    standings[home_id]["gf"] += sh
    standings[home_id]["ga"] += sa
    standings[away_id]["gf"] += sa
    standings[away_id]["ga"] += sh

    # points
    p_home, p_away = points_for_pair(sh, sa)
    standings[home_id]["pts"] += p_home
    standings[away_id]["pts"] += p_away

    # wins/draws/losses and pts_against (opponents' points vs this team)
    if sh > sa:  # home win
        standings[home_id]["wins"] += 1
        standings[away_id]["losses"] += 1
        standings[home_id]["pts_against"] += 0
        standings[away_id]["pts_against"] += 2
    elif sh < sa:  # away win
        standings[home_id]["losses"] += 1
        standings[away_id]["wins"] += 1
        standings[home_id]["pts_against"] += 2
        standings[away_id]["pts_against"] += 0
    else:  # draw
        standings[home_id]["draws"] += 1
        standings[away_id]["draws"] += 1
        standings[home_id]["pts_against"] += 1
        standings[away_id]["pts_against"] += 1

    # recompute gd explicitly
    standings[home_id]["gd"] = standings[home_id]["gf"] - standings[home_id]["ga"]
    standings[away_id]["gd"] = standings[away_id]["gf"] - standings[away_id]["ga"]

    # standings after THIS game
    tbl = standings_table_df()
    standing_after_home.append(tbl.loc[home_id, "rank"])
    standing_after_away.append(tbl.loc[away_id, "rank"])

    # push cumulative snapshots for both teams
    wins_home_after.append(standings[home_id]["wins"])
    draws_home_after.append(standings[home_id]["draws"])
    losses_home_after.append(standings[home_id]["losses"])
    pts_against_home_after.append(standings[home_id]["pts_against"])

    wins_away_after.append(standings[away_id]["wins"])
    draws_away_after.append(standings[away_id]["draws"])
    losses_away_after.append(standings[away_id]["losses"])
    pts_against_away_after.append(standings[away_id]["pts_against"])

# attach results to fixtures
df_all_fixtures_in_season["standing_home"] = standing_after_home
df_all_fixtures_in_season["standing_away"] = standing_after_away

df_all_fixtures_in_season["wins_home"] = wins_home_after
df_all_fixtures_in_season["draws_home"] = draws_home_after
df_all_fixtures_in_season["losses_home"] = losses_home_after
df_all_fixtures_in_season["pts_against_home"] = pts_against_home_after

df_all_fixtures_in_season["wins_away"] = wins_away_after
df_all_fixtures_in_season["draws_away"] = draws_away_after
df_all_fixtures_in_season["losses_away"] = losses_away_after
df_all_fixtures_in_season["pts_against_away"] = pts_against_away_after

# final standings (now includes wins/draws/losses/pts_against)
final_table = standings_table_df()
print("Final Standings:")
display(final_table[["rank","name","pts","pts_against","wins","draws","losses","gf","ga","gd"]])

# save to csv
df_all_fixtures_in_season.to_csv("fixtures.csv", index=False)

# save to duckdb
con.execute("DROP TABLE IF EXISTS fixtures_enhanced")
con.execute("CREATE TABLE fixtures_enhanced AS SELECT * FROM df_all_fixtures_in_season")

display((df_all_fixtures_in_season))


Final Standings:


,rank,name,pts,pts_against,wins,draws,losses,gf,ga,gd
fe7bdd16-3952-11ef-b585-af5c55c3771d,1,Füchse Berlin,58,10,27,4,3,1197,984,213
fe848316-3952-11ef-8185-af5c55c3771d,2,SC Magdeburg,57,11,28,1,5,1076,913,163
feb41b46-3952-11ef-9df6-af5c55c3771d,3,MT Melsungen,55,13,27,1,6,1020,906,114
febf038e-3952-11ef-b7c2-af5c55c3771d,4,THW Kiel,51,17,25,1,8,1061,946,115
fe88f93b-3952-11ef-aa5d-af5c55c3771d,5,SG Flensburg-Handewitt,47,21,21,5,8,1134,1028,106
fea93a14-3952-11ef-a7e0-af5c55c3771d,6,TSV Hannover-Burgdorf,44,24,20,4,10,1036,994,42
fe911367-3952-11ef-9131-af5c55c3771d,7,VfL Gummersbach,40,28,19,2,13,1051,1007,44
fe99a935-3952-11ef-9dd4-af5c55c3771d,8,TBV Lemgo Lippe,39,29,18,3,13,964,930,34
fe80598a-3952-11ef-914c-af5c55c3771d,9,Rhein-Neckar Löwen,36,32,17,2,15,1027,1027,0
0045fbc4-3953-11ef-a217-af5c55c3771d,10,Handball Sport Verein Hamburg,35,33,15,5,14,1060,1070,-10


,fixtureId,seasonId,fixtureNumber,nameLocal,nameLatin,startTimeLocal,startTimeUTC,roundNumber,externalId,entityId_home,...,standing_home,standing_away,wins_home,draws_home,losses_home,pts_against_home,wins_away,draws_away,losses_away,pts_against_away
0,ccdbc62f-4373-11ef-8506-0f9723377f57,cabcf509-4373-11ef-a370-9d3c1e90234a,1,TSV Hannover-Burgdorf vs. VfL Gummersbach,<NA>,2024-09-05T19:00:00,2024-09-05 17:00:00,1,57919,fea93a14-3952-11ef-a7e0-af5c55c3771d,...,18,1,0,0,1,2,1,0,0,0
1,cee2c85c-4373-11ef-94e2-cd77cb69a9df,cabcf509-4373-11ef-a370-9d3c1e90234a,2,TBV Lemgo Lippe vs. MT Melsungen,<NA>,2024-09-05T19:00:00,2024-09-05 17:00:00,1,57920,fe99a935-3952-11ef-9dd4-af5c55c3771d,...,18,1,0,0,1,2,1,0,0,0
2,d0f835ea-4373-11ef-a915-811a78587cbc,cabcf509-4373-11ef-a370-9d3c1e90234a,3,Rhein-Neckar Löwen vs. THW Kiel,<NA>,2024-09-05T20:30:00,2024-09-05 18:30:00,1,57921,fe80598a-3952-11ef-914c-af5c55c3771d,...,2,17,1,0,0,0,0,0,1,2
3,d1f5d36c-4373-11ef-ada8-811a78587cbc,cabcf509-4373-11ef-a370-9d3c1e90234a,4,FRISCH AUF! Göppingen vs. Handball Sport Verei...,<NA>,2024-09-06T19:00:00,2024-09-06 17:00:00,1,57922,fea4237d-3952-11ef-9fcd-af5c55c3771d,...,4,5,0,1,0,1,0,1,0,1
4,d3e126de-4373-11ef-9661-2f970b50aee5,cabcf509-4373-11ef-a370-9d3c1e90234a,5,SG Flensburg-Handewitt vs. HC Erlangen,<NA>,2024-09-06T20:00:00,2024-09-06 18:00:00,1,57923,fe88f93b-3952-11ef-aa5d-af5c55c3771d,...,1,18,1,0,0,0,0,0,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,a6d07af7-4374-11ef-b774-e12378ed22f2,cabcf509-4373-11ef-a370-9d3c1e90234a,<NA>,Rhein-Neckar Löwen vs. Füchse Berlin,<NA>,2025-06-08T15:00:00,2025-06-08 13:00:00,34,58220,fe80598a-3952-11ef-914c-af5c55c3771d,...,9,1,17,2,15,32,27,4,3,10
302,a79a0466-4374-11ef-8dc0-5f411ff1ae1b,cabcf509-4373-11ef-a370-9d3c1e90234a,<NA>,VfL Gummersbach vs. TSV Hannover-Burgdorf,<NA>,2025-06-08T15:00:00,2025-06-08 13:00:00,34,58222,fe911367-3952-11ef-9131-af5c55c3771d,...,7,6,19,2,13,28,20,4,10,24
303,a79b1a97-4374-11ef-b022-89956fae12b0,cabcf509-4373-11ef-a370-9d3c1e90234a,<NA>,TVB Stuttgart vs. SC DHfK Leipzig,<NA>,2025-06-08T15:00:00,2025-06-08 13:00:00,34,58221,febb3114-3952-11ef-b6f7-af5c55c3771d,...,16,13,9,0,25,50,10,1,23,47
304,a9648a25-4374-11ef-bba3-337ae13ae915,cabcf509-4373-11ef-a370-9d3c1e90234a,<NA>,1. VfL Potsdam vs. MT Melsungen,<NA>,2025-06-08T15:00:00,2025-06-08 13:00:00,34,58223,ffe84692-3952-11ef-954e-af5c55c3771d,...,18,3,3,0,31,62,27,1,6,13


### Get fixture

In [15]:
columns_to_keep_event_list = [
    # "clientId",
    # "clientType",
    "fixtureId",
    # "organizationId",
    # "received",
    # "sport",
    # "topic",
    # "type",
    "class",
    "eventId",
    "eventTime",
    "eventType",
    "subType",
    # "timestamp",
    "attendance",
    # "numberOfPeriods",
    # "periodLength",
    "entityId",
    "personId",
    # "status",
    # "active",
    "bib",
    # "captain",
    "name",
    "position",
    # "starter",
    # "number",
    "scores",
    "periodId",
    # "sequence",
    "playId",
    "clock",
    "success",
    "x",
    "y",
    "attackType",
    "goalKeeperId",
    "location",
    # "options",
    "failureReason",
    # "flagged",
    # "value",
    "emptyNet",
]


In [ ]:
# query fixtureIds from duckdb
list_fixture_ids = con.execute("SELECT fixtureId FROM fixtures").fetchall()
list_fixture_ids = [fid[0] for fid in list_fixture_ids]

print(f"Found {len(list_fixture_ids)} fixtures.")
df_all_players = pd.DataFrame()
df_all_match_events = pd.DataFrame()

for fid in list_fixture_ids:
    print(f"Downloading SETUP ONLY events for fixtureId: {fid}")

    match_events = api.get_fixture_events_by_id(
        fid, setup_only=False, with_scores=True
    )
    # print(f"  Events count: {len(setup_events)}")

    

    for event in match_events:
        if "data" in event and event["data"]:
            # Merge the data dict into the event dict
            event.update(event["data"])
            del event["data"]  # Remove the original data field
            pass
        if "options" in event and event["options"]:
            # Merge the options dict into the event dict
            event.update(event["options"])
            del event["options"]  # Remove the original options field

    df_match_events = pd.DataFrame(
        match_events, columns=columns_to_keep_event_list
    )
    

    # query teams from duckdb
    df_teams = con.execute("SELECT * FROM teams").df()
    # print(f"df_teams Teams count from duckdb: {len(df_teams)}")
    # display(df_teams)

    # insert namefulllocal to df_match_events from df_teams
    df_match_events = df_match_events.merge(
        df_teams[["entityId", "nameFullLocal"]],
        left_on="entityId",
        right_on="entityId",
        how="left",
    )
    # rename "nameFullLocal" column to "teamName"
    df_match_events = df_match_events.rename(
        columns={"nameFullLocal": "teamName"}
    )
    # filter for class == "setup" and eventType == "person"
    df_setup_events = df_match_events[
        (df_match_events["class"] == "setup")
        & (df_match_events["eventType"] == "person")
    ]
    # print(f"df_match_events Events count: {len(df_match_events)}")
    # display(df_match_events)

    # get list of unique personId's
    unique_person_ids = df_setup_events["personId"].unique()
    list_person_ids = unique_person_ids.tolist()
    # print(f"  Unique personIds count: {len(unique_person_ids)}")
    # get list of unique personId's
    unique_person_names = df_setup_events["name"].unique()
    # print(f"  Unique personNames count: {len(unique_person_names)}")
    # drop duplicates
    df_setup_events = df_setup_events.drop_duplicates(subset=["personId"])
    # print("df_match_events after filtering and drop_duplicates")
    

    
    display(df_match_events)

    str_person_ids = ",".join(list_person_ids)
    players = api.get_players_by_ids(person_ids=str_person_ids)
    df_players = pd.json_normalize(players)
    # print(f"  Downloaded player details count: {len(df_players)}")
    # display(df_players)
    # insert entityId to df_players from df_match_events

    people_map = (
        df_setup_events[["personId", "entityId", "teamName"]]
        .dropna(subset=["personId"])
        .drop_duplicates(subset=["personId"])
    )

    df_players = df_players.merge(
        people_map, on="personId", how="left", validate="one_to_one"  # ensures no duplication
    )
    # df_players = df_players.merge(
    #     df_match_events[["teamName", "personId", "entityId"]],
    #     left_on="personId",
    #     right_on="personId",
    #     how="left",
    # )
    print(f"Length of match events before personName insert: {len(df_match_events)}")
    # display(df_match_events)

    # insert nameFullLocal as personName to df_match_events from df_players
    df_match_events = df_match_events.merge(
        df_players[["nameFullLocal", "personId"]],
        left_on="personId",
        right_on="personId",
        how="left",
    )
    df_match_events = df_match_events.rename(columns={"nameFullLocal": "personName"})


    print(f"Length of match events before goalkeeperName insert: {len(df_match_events)}")
    # display(df_match_events)
    # insert nameFullLocal as personName to df_match_events from df_players
    df_match_events = df_match_events.merge(
        df_players[["nameFullLocal", "personId"]],
        left_on="goalKeeperId",
        right_on="personId",
        suffixes=("", "_goalkeeper"),
        how="left",
    )
    # drop column personId_goalkeeper
    df_match_events = df_match_events.rename(columns={"nameFullLocal": "goalkeeperName"})
    print("After inserting goalkeeperName")
    display(df_match_events)
    
    
    # print all cols in df_players
    # print(f"Columns in df_players: {df_players.columns.tolist()}")

    columns_to_keep_player_list = [
        # "added",
        # "deceased",
        "dob",
        "externalId",
        # "gender",
        # "historicalNames",
        "images",
        # "languageLocal",
        # "nameAbbreviated",
        "nameFamilyLatin",
        "nameFamilyLocal",
        "nameFullLatin",
        "nameFullLocal",
        "nameGivenLatin",
        "nameGivenLocal",
        "nationality",
        # "organizationId",
        "personId",
        # "representing",
        # "status",
        # "updated",
        "additionalDetails.height",
        "additionalDetails.weight",
        # "organization.id",
        # "organization.resourceType",
        "teamName",
        "entityId",
    ]

    df_players_filtered = df_players[
        [col for col in columns_to_keep_player_list if col in df_players.columns]
    ]
    # print("df_players_filtered after filtering")
    display(df_players_filtered)

    df_all_players = pd.concat(
        [df_all_players, df_players_filtered], ignore_index=True
    )
    print(f'" --> Length of all players collected: {len(df_all_players)}')

    break



Found 306 fixtures.


,fixtureId,class,eventId,eventTime,eventType,subType,attendance,entityId,personId,bib,...,clock,success,x,y,attackType,goalKeeperId,location,failureReason,emptyNet,teamName
0,00c08679-4374-11ef-80bd-73cf0bc66b45,setup,b7d8cd90-8ed2-11ef-b16d-cda25f1166cf,2024-10-20T11:01:55.049Z,fixture,post,9900.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00c08679-4374-11ef-80bd-73cf0bc66b45,setup,62917f30-8ed7-11ef-b16d-cda25f1166cf,2024-10-20T11:35:19.459Z,person,NaN,NaN,fe88f93b-3952-11ef-aa5d-af5c55c3771d,43e2dd60-3953-11ef-a0c7-855e48b08f9c,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SG Flensburg-Handewitt
2,00c08679-4374-11ef-80bd-73cf0bc66b45,setup,6591c6e0-8ed7-11ef-b16d-cda25f1166cf,2024-10-20T11:35:24.494Z,person,NaN,NaN,fe88f93b-3952-11ef-aa5d-af5c55c3771d,454487a1-3953-11ef-b79a-855e48b08f9c,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SG Flensburg-Handewitt
3,00c08679-4374-11ef-80bd-73cf0bc66b45,setup,6e1557a0-8ed7-11ef-b16d-cda25f1166cf,2024-10-20T11:35:38.778Z,person,NaN,NaN,fe88f93b-3952-11ef-aa5d-af5c55c3771d,45338f1d-3953-11ef-a2af-855e48b08f9c,24,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SG Flensburg-Handewitt
4,00c08679-4374-11ef-80bd-73cf0bc66b45,setup,708c9930-8ed7-11ef-b16d-cda25f1166cf,2024-10-20T11:35:42.915Z,person,NaN,NaN,fe88f93b-3952-11ef-aa5d-af5c55c3771d,0ef30fc3-3956-11ef-b635-63392ed267d4,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SG Flensburg-Handewitt
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
644,00c08679-4374-11ef-80bd-73cf0bc66b45,sport,a542b051-be35-11ef-9482-850ba4c8b178,2024-12-19T18:18:28.712Z,period,end,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
645,00c08679-4374-11ef-80bd-73cf0bc66b45,sport,a54b14c0-be35-11ef-9482-850ba4c8b178,2024-12-19T18:18:28.757Z,possession,NaN,NaN,fea93a14-3952-11ef-a7e0-af5c55c3771d,NaN,NaN,...,PT30M0S,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TSV Hannover-Burgdorf
646,00c08679-4374-11ef-80bd-73cf0bc66b45,sport,a5ddb7d0-be35-11ef-9482-850ba4c8b178,2024-12-19T18:18:29.709Z,period,confirmed,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
647,00c08679-4374-11ef-80bd-73cf0bc66b45,sport,a8177b30-be35-11ef-9482-850ba4c8b178,2024-12-19T18:18:33.443Z,fixture,end,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Length of match events before personName insert: 649
Length of match events before goalkeeperName insert: 649
After inserting goalkeeperName


,fixtureId,class,eventId,eventTime,eventType,subType,attendance,entityId,personId,bib,...,y,attackType,goalKeeperId,location,failureReason,emptyNet,teamName,personName,goalkeeperName,personId_goalkeeper
0,00c08679-4374-11ef-80bd-73cf0bc66b45,setup,b7d8cd90-8ed2-11ef-b16d-cda25f1166cf,2024-10-20T11:01:55.049Z,fixture,post,9900.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00c08679-4374-11ef-80bd-73cf0bc66b45,setup,62917f30-8ed7-11ef-b16d-cda25f1166cf,2024-10-20T11:35:19.459Z,person,NaN,NaN,fe88f93b-3952-11ef-aa5d-af5c55c3771d,43e2dd60-3953-11ef-a0c7-855e48b08f9c,1,...,NaN,NaN,NaN,NaN,NaN,NaN,SG Flensburg-Handewitt,Benjamin Burić,NaN,NaN
2,00c08679-4374-11ef-80bd-73cf0bc66b45,setup,6591c6e0-8ed7-11ef-b16d-cda25f1166cf,2024-10-20T11:35:24.494Z,person,NaN,NaN,fe88f93b-3952-11ef-aa5d-af5c55c3771d,454487a1-3953-11ef-b79a-855e48b08f9c,20,...,NaN,NaN,NaN,NaN,NaN,NaN,SG Flensburg-Handewitt,Kevin Møller,NaN,NaN
3,00c08679-4374-11ef-80bd-73cf0bc66b45,setup,6e1557a0-8ed7-11ef-b16d-cda25f1166cf,2024-10-20T11:35:38.778Z,person,NaN,NaN,fe88f93b-3952-11ef-aa5d-af5c55c3771d,45338f1d-3953-11ef-a2af-855e48b08f9c,24,...,NaN,NaN,NaN,NaN,NaN,NaN,SG Flensburg-Handewitt,Jim Gottfridsson,NaN,NaN
4,00c08679-4374-11ef-80bd-73cf0bc66b45,setup,708c9930-8ed7-11ef-b16d-cda25f1166cf,2024-10-20T11:35:42.915Z,person,NaN,NaN,fe88f93b-3952-11ef-aa5d-af5c55c3771d,0ef30fc3-3956-11ef-b635-63392ed267d4,25,...,NaN,NaN,NaN,NaN,NaN,NaN,SG Flensburg-Handewitt,Lukas Jørgensen,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
644,00c08679-4374-11ef-80bd-73cf0bc66b45,sport,a542b051-be35-11ef-9482-850ba4c8b178,2024-12-19T18:18:28.712Z,period,end,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
645,00c08679-4374-11ef-80bd-73cf0bc66b45,sport,a54b14c0-be35-11ef-9482-850ba4c8b178,2024-12-19T18:18:28.757Z,possession,NaN,NaN,fea93a14-3952-11ef-a7e0-af5c55c3771d,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,TSV Hannover-Burgdorf,NaN,NaN,NaN
646,00c08679-4374-11ef-80bd-73cf0bc66b45,sport,a5ddb7d0-be35-11ef-9482-850ba4c8b178,2024-12-19T18:18:29.709Z,period,confirmed,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
647,00c08679-4374-11ef-80bd-73cf0bc66b45,sport,a8177b30-be35-11ef-9482-850ba4c8b178,2024-12-19T18:18:33.443Z,fixture,end,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,dob,externalId,images,nameFamilyLatin,nameFamilyLocal,nameFullLatin,nameFullLocal,nameGivenLatin,nameGivenLocal,nationality,personId,additionalDetails.height,additionalDetails.weight,teamName,entityId
0,1994-01-17,27856,"[{'added': '2024-08-09T10:43:45', 'baseId': '0...",,Blagotinšek,,Blaž Blagotinšek,,Blaž,SVN,01be7e8c-3956-11ef-989c-2b303b141e24,202.0,120.0,SG Flensburg-Handewitt,fe88f93b-3952-11ef-aa5d-af5c55c3771d
1,2004-01-31,27995,[],Schmitt,Schmitt,Ben Kjell Schmitt,Ben Kjell Schmitt,Ben Kjell,Ben Kjell,DEU,0473473e-3956-11ef-ad70-2b303b141e24,197.0,135.0,SG Flensburg-Handewitt,fe88f93b-3952-11ef-aa5d-af5c55c3771d
2,1996-05-29,28060,"[{'added': '2024-08-09T10:44:09', 'baseId': '0...",Horgen,Horgen,Aksel Horgen,Aksel Horgen,Aksel,Aksel,NOR,0d97a4b4-3956-11ef-9a30-63392ed267d4,185.0,89.0,SG Flensburg-Handewitt,fe88f93b-3952-11ef-aa5d-af5c55c3771d
3,1998-01-14,3732,"[{'added': '2024-08-09T13:50:46', 'baseId': '0...",Stutzke,Stutzke,Lukas Stutzke,Lukas Stutzke,Lukas,Lukas,DEU,0deb05cf-3954-11ef-a6df-a3c22717a414,194.0,99.0,TSV Hannover-Burgdorf,fea93a14-3952-11ef-a7e0-af5c55c3771d
4,1999-03-31,28123,"[{'added': '2024-08-09T10:44:24', 'baseId': '0...",Jørgensen,Jørgensen,Lukas Jørgensen,Lukas Jørgensen,Lukas,Lukas,DNK,0ef30fc3-3956-11ef-b635-63392ed267d4,193.0,107.0,SG Flensburg-Handewitt,fe88f93b-3952-11ef-aa5d-af5c55c3771d
5,1996-03-01,28136,"[{'added': '2024-08-19T11:44:14', 'baseId': '0...",Strmljan,Strmljan,Tilen Strmljan,Tilen Strmljan,Tilen,Tilen,SVN,0f410352-3956-11ef-8c6a-63392ed267d4,185.0,77.0,TSV Hannover-Burgdorf,fea93a14-3952-11ef-a7e0-af5c55c3771d
6,1997-01-03,28137,"[{'added': '2024-08-09T13:48:32', 'baseId': '0...",Gade,Gade,Simon Gade,Simon Gade,Simon,Simon,DNK,0f46dff5-3956-11ef-bcc8-63392ed267d4,194.0,93.0,TSV Hannover-Burgdorf,fea93a14-3952-11ef-a7e0-af5c55c3771d
7,2000-12-11,28169,"[{'added': '2024-08-09T10:44:59', 'baseId': '0...",Pytlick,Pytlick,Simon Pytlick,Simon Pytlick,Simon,Simon,DNK,0feef743-3956-11ef-83b0-63392ed267d4,193.0,98.0,SG Flensburg-Handewitt,fe88f93b-3952-11ef-aa5d-af5c55c3771d
8,2006-02-09,28013,[],Czertowicz,Czertowicz,"Oskar, Czertowicz",Oskar Czertowicz,"Oskar,",Oskar,POL,19ac549b-3956-11ef-872c-1fafd93a56d6,NaN,NaN,SG Flensburg-Handewitt,fe88f93b-3952-11ef-aa5d-af5c55c3771d
9,2005-01-07,28497,"[{'added': '2024-10-24T13:56:35', 'baseId': '2...",Lutze,Lutze,Thorge Lutze,Thorge Lutze,Thorge,Thorge,DEU,20a0f2e8-3956-11ef-b0d2-f300d8784a69,188.0,97.0,TSV Hannover-Burgdorf,fea93a14-3952-11ef-a7e0-af5c55c3771d


" --> Length of all players collected: 41


In [ ]:
df_all_players_bak = df_all_players.copy()
df_all_match_events_bak = df_all_match_events.copy()

In [ ]:
print(f'" --> Length of all players collected: {len(df_all_players)}')
# remove duplicates in df_all_players based on personId and entityId, but show first
duplicates = df_all_players.duplicated(subset=["entityId","personId"], keep="first")
display(df_all_players[duplicates])
# drop duplicates
df_all_players = df_all_players.drop_duplicates(subset=["entityId","personId"], keep="first")
print(f'" --> Length of all players after dropping duplicates: {len(df_all_players)}')
display(df_all_players)
    # break

# drop table events if exists
con.execute("DROP TABLE IF EXISTS match_events")
# create if not exists table match_events in duckdb
con.execute("CREATE TABLE match_events AS SELECT * FROM df_all_match_events")
print("In Database:")
display(con.execute("SELECT * FROM match_events").df())

# drop table players if exists
con.execute("DROP TABLE IF EXISTS players")
# create if not exists table players in duckdb
con.execute("CREATE TABLE players AS SELECT * FROM df_all_players")
print("In Database:")
display(con.execute("SELECT * FROM players").df())
# Remove any columns not in the keep list
# df_match_events = pd.DataFrame(setup_events, columns=columns_to_keep_event_list)
# display(df_match_events)
# print(f"Columns: {df_match_events.columns.tolist()}")